In [0]:
RAW_PATH='/Volumes/workspace/default/healthcare_data_lake/'
BRONZE_PATH='/Volumes/workspace/default/healthcare_data_lake/bronze/'
SILVER_PATH='/Volumes/workspace/default/healthcare_data_lake/silver/'
GOLD_PATH='/Volumes/workspace/default/healthcare_data_lake/gold/'

#KEY OF GCP PROJECT
GCP_PROJECT='directed-truck-489818-t6'
BQ_QUERY='healthcare'
TEMP_GCS_BUCKET='healthcare-databricks-temp'


#secrets codes
GCP_SECRET_SCOPE='gcp-secret'
GCP_SECRET_KEY='gcp-sa-key'

In [0]:
diagnosis=spark.read.format('delta').load(BRONZE_PATH+'diagnosis')
labs=spark.read.format('delta').load(BRONZE_PATH+'labs')
outcomes=spark.read.format('delta').load(BRONZE_PATH+'outcomes')
patients=spark.read.format('delta').load(BRONZE_PATH+'patients')



In [0]:
patients.display()

patient_id,name,age,gender,diagnosis_id,admission_date,discharge_date,outcome_id,treatment_cost
1,Edward Montes,68,M,2,2025-03-22T00:00:00.000Z,2025-04-04T00:00:00.000Z,1,4443.0
2,Jessica Zimmerman,81,M,2,2025-03-31T00:00:00.000Z,2025-04-02T00:00:00.000Z,2,4265.0
3,Gina Alexander,58,M,6,2025-01-08T00:00:00.000Z,2025-03-13T00:00:00.000Z,1,4188.0
4,Lori Green,44,F,3,2025-04-03T00:00:00.000Z,2025-04-05T00:00:00.000Z,1,9783.0
5,Jack Logan,72,F,9,2025-06-03T00:00:00.000Z,2025-06-04T00:00:00.000Z,1,5005.0
6,Melinda Robinson,37,F,4,2025-03-31T00:00:00.000Z,2025-04-05T00:00:00.000Z,1,5281.0
7,Sherry Robinson,50,F,1,2025-06-11T00:00:00.000Z,2025-07-05T00:00:00.000Z,2,2984.0
8,James Watts,68,M,4,2025-04-25T00:00:00.000Z,2025-06-19T00:00:00.000Z,1,9136.0
9,Jeanne Butler,87,F,1,2025-04-15T00:00:00.000Z,2025-04-17T00:00:00.000Z,2,9790.0
10,Benjamin Rogers,48,M,5,2025-05-15T00:00:00.000Z,2025-05-19T00:00:00.000Z,2,3639.0


In [0]:
from pyspark.sql.functions import col, to_date
from pyspark.sql.types import DoubleType
patients_clean=patients.dropDuplicates(['patient_id'])\
    .withColumn('admission_date', to_date(col('admission_date'), 'dd_MM_yyyy'))\
    .withColumn('discharge_date', to_date(col('discharge_date'), 'dd_MM_yyyy'))\
    .withColumn('treatment_cost', col('treatment_cost').cast(DoubleType()))

patients_clean.display()
patients_clean.write.format("delta").mode("overwrite").save(SILVER_PATH+"patients_clean")

patient_id,name,age,gender,diagnosis_id,admission_date,discharge_date,outcome_id,treatment_cost
1,Edward Montes,68,M,2,2025-03-22,2025-04-04,1,4443.0
2,Jessica Zimmerman,81,M,2,2025-03-31,2025-04-02,2,4265.0
3,Gina Alexander,58,M,6,2025-01-08,2025-03-13,1,4188.0
4,Lori Green,44,F,3,2025-04-03,2025-04-05,1,9783.0
5,Jack Logan,72,F,9,2025-06-03,2025-06-04,1,5005.0
6,Melinda Robinson,37,F,4,2025-03-31,2025-04-05,1,5281.0
7,Sherry Robinson,50,F,1,2025-06-11,2025-07-05,2,2984.0
8,James Watts,68,M,4,2025-04-25,2025-06-19,1,9136.0
9,Jeanne Butler,87,F,1,2025-04-15,2025-04-17,2,9790.0
10,Benjamin Rogers,48,M,5,2025-05-15,2025-05-19,2,3639.0


In [0]:
labs.display()

lab_id,patient_id,test_name,result,normal_range
1,196,Blood Pressure,89.7,150-200
2,165,Cholesterol,270.2,13-17
3,182,Blood Pressure,268.0,0.6-1.2
4,142,Cholesterol,57.3,150-200
5,55,Blood Sugar,256.5,120/80
6,176,Creatinine,82.2,120/80
7,7,Blood Pressure,133.8,150-200
8,74,Hemoglobin,235.9,0.6-1.2
9,7,Hemoglobin,90.2,20-50
10,161,Vitamin D,254.5,13-17


In [0]:
from pyspark.sql import functions as F
labs_clean = labs.withColumn(
    "normal_range",
    F.expr("""
        concat(
            split(regexp_replace(normal_range,'/','-'),'-')[1],
            '-',
            split(regexp_replace(normal_range,'/','-'),'-')[0]
        )
    """)
    
)

labs_clean.display()
labs_clean.write.mode('overwrite').format('delta').save(SILVER_PATH+"labs_clean")



lab_id,patient_id,test_name,result,normal_range
1,196,Blood Pressure,89.7,200-150
2,165,Cholesterol,270.2,17-13
3,182,Blood Pressure,268.0,1.2-0.6
4,142,Cholesterol,57.3,200-150
5,55,Blood Sugar,256.5,80-120
6,176,Creatinine,82.2,80-120
7,7,Blood Pressure,133.8,200-150
8,74,Hemoglobin,235.9,1.2-0.6
9,7,Hemoglobin,90.2,50-20
10,161,Vitamin D,254.5,17-13


In [0]:
outcomes.display()
outcomes_clean=outcomes.dropDuplicates(['outcome_id'])
outcomes_clean.write.mode('overwrite').format('delta').save(SILVER_PATH+"outcomes_clean")


outcome_id,outcome_name
1,Recovered
2,Complicated
3,Deceased


In [0]:
diagnosis.display()
diagnosis_clean=diagnosis.dropDuplicates(['diagnosis_id'])
diagnosis_clean.write.mode('overwrite').format('delta').save(SILVER_PATH+"diagnosis_clean")

diagnosis_id,diagnosis_name
1,Hypertension
2,Diabetes
3,Heart Disease
4,Asthma
5,Stroke
6,COPD
7,Cancer
8,Arthritis
9,Kidney Disease
10,Liver Disease


In [0]:
silver_table=patients_clean.alias('p')\
    .join(labs_clean.alias('l'), 'patient_id', 'left')\
    .join(diagnosis_clean.alias('d'), 'diagnosis_id', 'left')\
    .join(outcomes_clean.alias('o'), 'outcome_id', 'left')


        
silver_table.display()
silver_table.write.mode('overwrite').format('delta').save(SILVER_PATH+"silver_table")

outcome_id,diagnosis_id,patient_id,name,age,gender,admission_date,discharge_date,treatment_cost,lab_id,test_name,result,normal_range,diagnosis_name,outcome_name
2,8,12,Stacy Clark,40,F,2025-04-24,2025-05-18,4730.0,174,Creatinine,174.5,50-20,Arthritis,Complicated
2,9,70,James Leblanc,76,M,2025-02-11,2025-03-02,3625.0,64,Cholesterol,118.3,50-20,Kidney Disease,Complicated
1,4,161,Ann Adams,56,M,2025-01-16,2025-01-22,8305.0,10,Vitamin D,254.5,17-13,Asthma,Recovered
3,2,186,Melissa Torres,78,F,2025-04-25,2025-04-30,9218.0,58,Creatinine,248.7,50-20,Diabetes,Deceased
1,3,190,Kristin Underwood,59,F,2025-02-25,2025-06-26,7383.0,129,Blood Pressure,131.6,200-150,Heart Disease,Recovered
2,1,16,Olivia Roy,65,F,2025-06-13,2025-06-15,9151.0,183,Blood Sugar,68.0,1.2-0.6,Hypertension,Complicated
2,10,64,Rachel Johnson,38,F,2025-05-31,2025-06-03,9833.0,98,Vitamin D,81.3,1.2-0.6,Liver Disease,Complicated
3,7,74,William Mclaughlin,37,M,2025-05-13,2025-05-19,5281.0,158,Hemoglobin,254.3,50-20,Cancer,Deceased
3,6,138,Jessica Watson,62,F,2025-06-22,2025-07-02,8733.0,91,Blood Sugar,82.8,120-70,COPD,Deceased
3,7,146,Jessica Long,36,M,2025-01-20,2025-04-18,4161.0,65,Hemoglobin,287.7,17-13,Cancer,Deceased


In [0]:
from pyspark.sql.functions import when

patients_dq = patients_clean.withColumn(
    "dq_status",
    when(col("age") < 0, "INVALID_AGE")
    .when(col("age") > 120, "INVALID_AGE")
    .when(col("treatment_cost") <= 0, "INVALID_COST")
    .when(col("patient_id").isNull(), "NULL_PATIENT_ID")
    .otherwise("VALID")
)


patients_dq.display()

patients_dq.write.mode('overwrite').format('delta').save(SILVER_PATH+"patients_dq")

patient_id,name,age,gender,diagnosis_id,admission_date,discharge_date,outcome_id,treatment_cost,dq_status
1,Edward Montes,68,M,2,2025-03-22,2025-04-04,1,4443.0,VALID
2,Jessica Zimmerman,81,M,2,2025-03-31,2025-04-02,2,4265.0,VALID
3,Gina Alexander,58,M,6,2025-01-08,2025-03-13,1,4188.0,VALID
4,Lori Green,44,F,3,2025-04-03,2025-04-05,1,9783.0,VALID
5,Jack Logan,72,F,9,2025-06-03,2025-06-04,1,5005.0,VALID
6,Melinda Robinson,37,F,4,2025-03-31,2025-04-05,1,5281.0,VALID
7,Sherry Robinson,50,F,1,2025-06-11,2025-07-05,2,2984.0,VALID
8,James Watts,68,M,4,2025-04-25,2025-06-19,1,9136.0,VALID
9,Jeanne Butler,87,F,1,2025-04-15,2025-04-17,2,9790.0,VALID
10,Benjamin Rogers,48,M,5,2025-05-15,2025-05-19,2,3639.0,VALID
